# 03 — Delta-tabellen rechtstreeks lezen (zonder Trino)

Soms wil je voorbij Trino: kijken naar partitionering, time-travel doen,
of een Delta-tabel uit een notebook schrijven. Daarvoor hebben we
`delta-rs` (Rust-implementatie, Python-bindings) in de kernel zitten.

De credentials zijn de MinIO-keys uit `minio-s3-credentials` — die geven
rauwe S3-toegang. **Let op:** dit gaat *buiten* OPA om. Vraag jezelf af
of dat past bij wat je doet.

In [ ]:
from uwv_lab import s3

# Lees een silver-tabel rechtstreeks van MinIO als pandas DataFrame.
df = s3.read_delta('s3a://uwv-silver/uwv/persona_created/')
df.head()

In [ ]:
# Schema + versiehistorie van de Delta-tabel.
from deltalake import DeltaTable
from uwv_lab.s3 import _storage_options

dt = DeltaTable('s3a://uwv-silver/uwv/persona_created/', storage_options=_storage_options())
print('Schema:')
print(dt.schema().to_pyarrow())
print('\nLast 5 versies:')
for entry in dt.history()[:5]:
    print(entry)

In [ ]:
# Time-travel: lees versie N
import pandas as pd
from deltalake import DeltaTable
from uwv_lab.s3 import _storage_options

dt = DeltaTable('s3a://uwv-silver/uwv/persona_created/',
                version=0,
                storage_options=_storage_options())
pd.DataFrame(dt.to_pyarrow_table().to_pandas()).head()

## Schrijven

Schrijven naar `bronze` / `silver` / `gold` is voor de meeste rollen niet
toegestaan (zie OPA-policy + MinIO-bucket-policy). De
`uwv-staging`-bucket bestaat juist voor experimenteren — onderstaande
cel gebruikt die zone.

In [ ]:
import pandas as pd
from uwv_lab import s3

scratch = pd.DataFrame({'a': [1, 2, 3], 'b': ['x', 'y', 'z']})
s3.write_delta(scratch, 's3a://uwv-staging/lab/my_first_table/', mode='overwrite')
print('Geschreven. Verifieer met:')
print(s3.read_delta('s3a://uwv-staging/lab/my_first_table/'))